<div dir="rtl" align="right">

# آزمون فرض ۴

## وارد کردن کتابخانه‌ها

</div>

In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu

<div dir="rtl" align="right">

## خواندن داده

داده‌ی تمیز شده خوانده می‌شود، سپس **فقط تهران** و **فقط فروش** نگه داشته می‌شود.  
بعد از آن **قیمت** تمیز می‌شود: مقادیر خالی و غیرمثبت حذف می‌شوند و داده‌های خارج از **صدک ۱ تا ۹۹** کنار گذاشته می‌شوند.

</div>

In [ ]:
# خوندن دیتا
df = pd.read_csv(r"C:\Users\Asus\Desktop\final_data_slim.csv", low_memory=False)

# فقط تهران
df = df[df["city_slug"] == "tehran"].copy()

# فقط فروش
df = df[df["deal_family"] == "sell"].copy()

# قیمت رو تمیز می‌کنیم
price = "price_value"
df = df[df[price].notna() & (df[price] > 0)]
low, high = df[price].quantile([0.01, 0.99])
df = df[(df[price] >= low) & (df[price] <= high)]

<div dir="rtl" align="right">

## تابع آزمون من‌ویتنی

این تابع قیمت را بین دو گروه **دارای ویژگی** و **بدون ویژگی** مقایسه می‌کند:

1. انجام **آزمون من‌ویتنی**
2. محاسبه **میانه** هر گروه و **درصد تفاوت** میانه‌ها
3. چاپ نتایج آزمون
4. **تفسیر خودکار** بر اساس پی‌ولیو (آلفا `0.05`) و اندازه‌ی تفاوت:
   - تفاوت **بزرگ**: بیشتر یا مساوی ۱۰۰٪
   - تفاوت **متوسط**: بیشتر یا مساوی ۳۰٪
   - تفاوت **کوچک**: بیشتر یا مساوی ۱۰٪
   - تفاوت **ناچیز**: کمتر از ۱۰٪

</div>

In [ ]:
# تابع آزمون من ویتنی
def mw_test(df, col, name, price_col=price):
    sub = df[df[col].notna()]
    g1 = sub.loc[sub[col] == True, price_col]
    g0 = sub.loc[sub[col] == False, price_col]

    # آزمون من ویتنی
    u, p = mannwhitneyu(g1, g0, alternative="two-sided")

    # میانه و درصد تفاوت
    med1, med0 = g1.median(), g0.median()
    diff = (med1 - med0) / med0 * 100

    # چاپ نتایج آزمون
    print(f"\n{name}")
    print(f"  n_with    = {len(g1):,}")
    print(f"  n_without = {len(g0):,}")
    print(f"  med_with    = {med1:,.0f}")
    print(f"  med_without = {med0:,.0f}")
    print(f"  diff = {diff:+.1f}%")
    print(f"  p    = {p:.3g}")

    # تفسیر خودکار
    if p < 0.05:
        if abs(diff) >= 100:
            print("  نتیجه: تفاوت معنادار و بزرگ")
        elif abs(diff) >= 30:
            print("  نتیجه: تفاوت معنادار و متوسط")
        elif abs(diff) >= 10:
            print("  نتیجه: تفاوت معنادار ولی کوچک")
        else:
            print("  نتیجه: تفاوت معنادار ولی ناچیز")
    else:
        print("  نتیجه: تفاوت غیرمعنادار")
    print()

<div dir="rtl" align="right">

## آزمون استخر

بررسی تأثیر داشتن **استخر** (ویژگی لاکچری) بر قیمت فروش.

</div>

In [ ]:
# آزمون استخر
mw_test(df, "is_pool_tehran", "استخر (لاکچری)")

<div dir="rtl" align="right">

## آزمون پارکینگ

بررسی تأثیر داشتن **پارکینگ** (ویژگی غیرلاکچری) بر قیمت فروش.

</div>

In [ ]:
# آزمون پارکینگ
mw_test(df, "is_parking_tehran", "پارکینگ (غیرلاکچری)")